# 01 — Explore Qwen3.5-0.8B's architecture

**Goal of this notebook:** before we do any "surgery" on this model (drop the generation head, add a pointer head, etc.), we need to actually *see* what we're cutting into. No guessing from the README prose.

Specifically we want to answer, from the real loaded model, not from documentation:
1. What are the top-level submodules? (vision tower vs language backbone vs output head)
2. Does the repeating block pattern from the model card — "6 blocks of [3x Gated DeltaNet -> FFN, then 1x Gated Attention -> FFN]" — actually show up as 24 distinct layer objects?
3. How are the parameters distributed (vision vs language vs head)? Does it add up to ~0.8B?
4. Is the output head really tied to the input embedding, as the card claims?

Don't just run every cell and move on — read the printed output before running the next cell, and try to answer the question posed in each markdown block in your own words (a comment in the cell below it is fine) before continuing.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["HF_HUB_DISABLE_XET"] = "1"

In [2]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

MODEL_ID = "Qwen/Qwen3.5-0.8B"

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

CUDA available: True
Device: NVIDIA GeForce RTX 5060 Laptop GPU


## Loading the processor and the model

Two different objects, two different jobs:

- **`AutoProcessor`** is not a neural network. It's the glue code that turns raw inputs (a PIL image, a text string) into the tensors the model actually consumes: it resizes/normalizes images into pixel tensors, and tokenizes text into integer IDs. It also knows the model's *chat template* (the exact string format the model was trained on, e.g. where `<|im_start|>` goes).
- **`AutoModelForMultimodalLM`** is the actual neural network: the weights, the forward pass, everything that does matrix multiplication.

First run of the next cell will download the weights from the Hugging Face Hub (small model, should be a couple GB) and cache them under `~/.cache/huggingface`. Subsequent runs reuse the cache.

We load weights in `bfloat16` rather than the default `float32`. Why: your GPU (Blackwell) has native, fast bf16 support, it halves VRAM vs fp32, and bf16 has the same exponent range as fp32 (just less mantissa precision) so it doesn't suffer the overflow problems float16 historically had with transformers. `device_map="auto"` lets `accelerate` decide which device each weight shard goes on — with a 0.8B model and an 8GB GPU, it should all fit on `cuda:0`.

In [3]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

print(type(processor))
print(type(model))
print("A sample weight lives on:", next(model.parameters()).device, next(model.parameters()).dtype)

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

<class 'transformers.models.qwen3_vl.processing_qwen3_vl.Qwen3VLProcessor'>
<class 'transformers.models.qwen3_5.modeling_qwen3_5.Qwen3_5ForConditionalGeneration'>
A sample weight lives on: cuda:0 torch.bfloat16


## Question 1: what does the module tree look like?

`print(model)` on any PyTorch `nn.Module` recursively prints every submodule's class name and the submodules inside it. For a model this deep the output will be long — that's fine, scroll it.

**Before running:** guess how many top-level attributes you'd expect (e.g. something vision-related, something language-related, something for output). After running, scroll to the top of the output and compare against your guess.

In [ ]:
print(model)

**Answer in a comment below:** what are the top-level named children of `model` (e.g. `model.visual`, `model.model`, `model.lm_head`...)? What class name does the vision submodule have? What class name does the language-model submodule have?

In [4]:
for name, child in model.named_children():
    print(f"{name:30s} {type(child).__name__}")

model                          Qwen3_5Model
lm_head                        Linear


In [5]:
for name, child in model.named_children():
    print(f"{name:30s} {type(child).__name__}")
    if len(list(child.children())) > 0:
        for name2, child2 in child.named_children():
            print(f"{name:30s} {type(child).__name__} -> {name2:30s} {type(child2).__name__}")

model                          Qwen3_5Model
model                          Qwen3_5Model -> visual                         Qwen3_5VisionModel
model                          Qwen3_5Model -> language_model                 Qwen3_5TextModel
lm_head                        Linear


## Question 2: finding the repeating hybrid block pattern

The model card described the language backbone as **6 blocks of (3x Gated DeltaNet -> FFN, then 1x Gated Attention -> FFN)** = 24 layers total. `print(model)` already showed you this, but it's easy to miss in a wall of text. Let's extract it programmatically instead: walk every layer in the language model's layer stack and print just its class name and index.

You'll need to adjust the attribute path below (`model.model.layers` is a common pattern in HF causal LMs, but confirm it matches what you saw in the `named_children()` output above — the attribute might be nested one level differently for this model).

In [6]:
model.get_input_embeddings()

Embedding(248320, 1024)

In [9]:
model.get_output_embeddings()

Linear(in_features=1024, out_features=248320, bias=False)

In [10]:
# Adjust this path based on what you found above.
layers = model.model.language_model.layers
print(f"Number of layers: {len(layers)}")
for i, layer in enumerate(layers):
    # A layer is itself a container of submodules (attention/mixer + FFN).
    # Print its direct children's class names to see which kind of mixer it uses.
    mixer_classes = [type(child).__name__ for _, child in layer.named_children()]
    print(f"layer {i:2d}: {mixer_classes}")

Number of layers: 24
layer  0: ['Qwen3_5GatedDeltaNet', 'Qwen3_5MLP', 'Qwen3_5RMSNorm', 'Qwen3_5RMSNorm']
layer  1: ['Qwen3_5GatedDeltaNet', 'Qwen3_5MLP', 'Qwen3_5RMSNorm', 'Qwen3_5RMSNorm']
layer  2: ['Qwen3_5GatedDeltaNet', 'Qwen3_5MLP', 'Qwen3_5RMSNorm', 'Qwen3_5RMSNorm']
layer  3: ['Qwen3_5Attention', 'Qwen3_5MLP', 'Qwen3_5RMSNorm', 'Qwen3_5RMSNorm']
layer  4: ['Qwen3_5GatedDeltaNet', 'Qwen3_5MLP', 'Qwen3_5RMSNorm', 'Qwen3_5RMSNorm']
layer  5: ['Qwen3_5GatedDeltaNet', 'Qwen3_5MLP', 'Qwen3_5RMSNorm', 'Qwen3_5RMSNorm']
layer  6: ['Qwen3_5GatedDeltaNet', 'Qwen3_5MLP', 'Qwen3_5RMSNorm', 'Qwen3_5RMSNorm']
layer  7: ['Qwen3_5Attention', 'Qwen3_5MLP', 'Qwen3_5RMSNorm', 'Qwen3_5RMSNorm']
layer  8: ['Qwen3_5GatedDeltaNet', 'Qwen3_5MLP', 'Qwen3_5RMSNorm', 'Qwen3_5RMSNorm']
layer  9: ['Qwen3_5GatedDeltaNet', 'Qwen3_5MLP', 'Qwen3_5RMSNorm', 'Qwen3_5RMSNorm']
layer 10: ['Qwen3_5GatedDeltaNet', 'Qwen3_5MLP', 'Qwen3_5RMSNorm', 'Qwen3_5RMSNorm']
layer 11: ['Qwen3_5Attention', 'Qwen3_5MLP', 'Qwen3_

**Check:** does the pattern you see repeat every 4 layers as `[DeltaNet-ish, DeltaNet-ish, DeltaNet-ish, Attention-ish]` x 6? If the class names don't obviously say "DeltaNet" or "Attention", look at the attribute name of the mixer submodule inside one layer (e.g. `layer.self_attn` vs `layer.linear_attn` vs `layer.mixer`) — print `layers[0]` and `layers[3]` in full (just `print(layers[0])`) and diff them by eye if needed.

## Question 3: where do the parameters actually live?

"0.8B" is a marketing-ish headline number. Let's verify it and see the split between the vision tower, the language backbone, and the output head. This matters later: when we do model surgery, we'll likely want to freeze big chunks (e.g. the vision tower) and only train a small adapter, so knowing where the weight mass sits tells us where the training cost sits.

In [14]:
from collections import defaultdict

total = 0
by_top_level = defaultdict(int)

for name, p in model.named_parameters():
    n = p.numel()
    total += n
    top = ".".join(name.split(".")[:2])
    by_top_level[top] += n

print(f"Total parameters: {total:,}  (~{total/1e9:.3f}B)\n")
for top, n in sorted(by_top_level.items(), key=lambda kv: -kv[1]):
    print(f"{top:20s} {n:>15,}  ({n/total*100:5.1f}%)")

Total parameters: 852,985,920  (~0.853B)

model.language_model     752,393,024  ( 88.2%)
model.visual             100,592,896  ( 11.8%)


**Answer in a comment:** which top-level component holds the most parameters — vision, language layers, or embeddings/head? Does the total roughly match the "0.8B" in the model's name? (It likely won't match exactly — the name typically refers to just one part, e.g. the language model, not vision+language combined. Worth confirming which.)

## Question 4: is the output head really tied to the input embedding?

The model card claims "LM output tied to the embedding" — meaning the matrix that turns a token ID into a vector (input embedding) and the matrix that turns a final hidden state back into vocabulary logits (output/LM head) are literally the *same* tensor in memory, not just the same shape. This is a common parameter-saving trick. Let's verify it rather than trust the prose: if two tensors are tied, they should be the exact same Python object (`is`), not merely equal in value.

In [15]:
input_embed = model.get_input_embeddings()
output_embed = model.get_output_embeddings()

print("input embedding weight shape:", input_embed.weight.shape)
print("output head weight shape:", output_embed.weight.shape if output_embed is not None else None)
print("Are they literally the same tensor object?", input_embed.weight is output_embed.weight if output_embed is not None else "no output head found")

input embedding weight shape: torch.Size([248320, 1024])
output head weight shape: torch.Size([248320, 1024])
Are they literally the same tensor object? True


## Wrap-up

Write a short summary for yourself (a markdown cell below is fine) answering:
- The attribute path to the vision tower and to the language layer stack.
- Confirmation (or correction) of the 6x[3 DeltaNet + 1 Attention] pattern.
- The vision/language/head parameter split.
- Whether input/output embeddings are tied.

We'll need all four of these facts in the next notebook, where we run an actual forward pass and inspect hidden state shapes at a Gated DeltaNet layer vs a Gated Attention layer — that's where the "why can't we use block-causal packing on a hybrid model" question from any2jev's README gets a concrete, hands-on answer instead of a one-line claim.